In [1]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw
parse_tech, colors, tech_order, tech_colors = sw.template.get() # Template

In [2]:
model_inputs_path = '../../model/inputs/'
model_outputs_path = '../../model/outputs/'
gen_per_res_vs_marg_cost = '../../data/XM-API/Plan/gen_per_res_vs_marg_cost/Esc0.csv'
emissions = '../../data/XM-API/Plan/emissions/esc0.csv'
cap_2023 = '../../data/XM-API/Plan/installed_capacity/esc0/2023.csv'
cap_2037 = '../../data/XM-API/Plan/installed_capacity/esc0/2037.csv'
dema_path = '../../data/XM-API/variable_query/2022-12-01_2023-11-30/'

In [3]:

gen_build_costs=pd.read_csv(model_inputs_path+'gen_build_costs.csv')
gen_build_costs = gen_build_costs[gen_build_costs['build_year'] <= 2023]

gen_info=pd.read_csv(model_inputs_path+'gen_info.csv')
gen_build_costs = pd.merge(gen_build_costs, gen_info, on='GENERATION_PROJECT')

gen_build_costs['gen_tech'] = gen_build_costs['gen_tech'].replace(parse_tech)
gen_costs = gen_build_costs[['gen_tech','gen_overnight_cost','gen_fixed_om','gen_variable_om']]
print(gen_costs.head())

  gen_tech  gen_overnight_cost  gen_fixed_om  gen_variable_om
0  Thermal             3076000        143220         8.391108
1  Thermal             1420000        143220         8.391108
2  Thermal             3076000        143220         8.391108
3  Thermal             1421000        143220         8.391108
4  Thermal             1254000        143220         7.080000


In [4]:
gen_overnight_cost  = gen_costs.copy()
gen_overnight_cost = gen_overnight_cost[['gen_tech','gen_overnight_cost']]

gen_overnight_cost = gen_overnight_cost.groupby(['gen_tech']).agg({'gen_overnight_cost':'mean'}).reset_index()

fig = px.bar(
    gen_overnight_cost, x='gen_tech', y='gen_overnight_cost', 
    color_discrete_sequence=[colors[0]],
    labels={'gen_tech': 'Generation Technology', 'gen_overnight_cost': 'Mean Overnight Cost'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
fig.show()

In [5]:
gen_fixed_om  = gen_costs.copy()
gen_fixed_om = gen_fixed_om[['gen_tech','gen_fixed_om']]

gen_fixed_om = gen_fixed_om.groupby(['gen_tech']).agg({'gen_fixed_om':'mean'}).reset_index()

fig = px.bar(
    gen_fixed_om, x='gen_tech', y='gen_fixed_om',
    color_discrete_sequence=[colors[0]],
    labels={'gen_tech': 'Generation Technology', 'gen_fixed_om': 'Mean Fixed O&M Costs'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
fig.show()

In [6]:
gen_variable_om  = gen_costs.copy()
gen_variable_om = gen_variable_om[['gen_tech','gen_variable_om']]

gen_variable_om = gen_variable_om.groupby(['gen_tech']).agg({'gen_variable_om':'mean'}).reset_index()

fig = px.bar(
    gen_variable_om, x='gen_tech', y='gen_variable_om',
    color_discrete_sequence=[colors[0]],
    labels={'gen_tech': 'Generation Technology', 'gen_variable_om': 'Mean Variable O&M Costs'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
fig.show()

In [ ]:
fuel_cost=pd.read_csv(model_inputs_path+'fuel_cost.csv')

# Calcular el promedio del costo por tipo de combustible
fuel_cost = fuel_cost.groupby('fuel')['fuel_cost'].mean().reset_index()
fuel_cost['fuel'] = fuel_cost['fuel'].replace(
    {
        'ACPM': 'Diesel',
        'CARBON': 'Coal',
        'COMBUSTOLEO': 'Fuel Oil',
        'GASIMPOR': 'Imported Gas',
        'GASNACIO': 'National Gas'
    })

fig = px.bar(
    fuel_cost, x='fuel', y='fuel_cost',
    color_discrete_sequence=[colors[0]],
    labels={'fuel': 'Fuel', 'fuel_cost': 'Price per Basic Unit (USD)'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
fig.write_image("../images/price_per_basic_unit.png")
fig.show()

: 

In [8]:
model_path = '../../../model/inputs/'
fuels = pd.read_csv(model_inputs_path+'fuels.csv')
fuels = fuels[['fuel','co2_intensity']]

# Lista de valores de interés
filtered_fuels = ['GASNACIO', 'ACPM', 'CARBON', 'COMBUSTOLEO', 'GLP']

# Filtra el DataFrame donde los valores de 'columna_deseada' están en la lista de interés
fuels = fuels[fuels['fuel'].isin(filtered_fuels)]
fuels['fuel'] = fuels['fuel'].str.replace('GASNACIO', 'Natural Gas')
fuels['fuel'] = fuels['fuel'].str.replace('GASIMPOR', 'Imported Gas')
fuels['fuel'] = fuels['fuel'].str.replace('ACPM', 'Diesel')
fuels['fuel'] = fuels['fuel'].str.replace('CARBON', 'Coal')
fuels['fuel'] = fuels['fuel'].str.replace('COMBUSTOLEO', 'Fuel Oil')
fuels['fuel'] = fuels['fuel'].str.replace('GLP', 'LPG')
import plotly.express as px
# Crear la gráfica de barras, usando 'gen_tech' como color
fig = px.bar(
    fuels, x='fuel', y='co2_intensity',
    title='CO2 Intensity per Technology',
    color_discrete_sequence=[colors[0]],
    labels={'co2_intensity': 'CO2 Intensity (tCO2/MMBTU)', 'fuel': 'Fuel Type'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
# Mostrar la gráfica
fig.show()